# **Assignment 3: Create Chatbot using Transformer**

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import time
import psutil
import os

# 1. Configuration & Data Saving (Task 1)
model_name = "microsoft/DialoGPT-small"
local_save_path = "./my_local_model"

print(">> System: Initializing Transformer Engine...")
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=local_save_path)
model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir=local_save_path)

# Resource Tracker setup
process = psutil.Process(os.getpid())

chat_history_ids = None

def chat():
    global chat_history_ids
    print("\nChatbot: Hello! I am your AI assistant. How can I help you today? (Type 'exit' to quit)") #

    while True:
        # 2. User Input Handling 
        user_input = input("User: ")
        print(f"User: {user_input}")    # Adding this line specifically for .ipynb

        # 5. Exit condition (Task 5)
        if user_input.lower() in ['exit', 'quit']: #
            print("ChatBot: Goodbye!")
            break

        # Start Resource Tracking
        start_time = time.perf_counter()
        start_mem = process.memory_info().rss / (1024 * 1024)

        # 3. Encoding & Response Generation (Task 3)
        # Tokenize user input
        inputs = tokenizer(user_input + tokenizer.eos_token, return_tensors='pt')
        new_user_input_ids = inputs['input_ids']

        # 4. Continuous Conversation Management (Task 4)
        # Append new input to history
        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
        else:
            bot_input_ids = new_user_input_ids

        full_attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

        # Got below parameters from Gemini custom for my PC specs. (i5-1235U CPU, 20GB RAM and 10GB integrate GPU)
        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=full_attention_mask,
            max_new_tokens=50,             # Booster: Prevents 3-minute wait times
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,        # Logic: Prevents "xD xD xD" loops
            do_sample=True,
            top_k=40,                      # Efficiency: Reduces CPU sorting burden
            top_p=0.9,
            temperature=0.6,               # Quality: Keeps response factual, not "creepy"
            repetition_penalty=1.2,        # Added: Extra layer against loops
            use_cache=True                 # Efficiency: Reuses math for faster turns
        )

        # End Resource Tracking
        end_time = time.perf_counter()
        end_mem = process.memory_info().rss / (1024 * 1024)

        # Display Output
        # Extract only the newly generated tokens from the end of the history
        response_ids = chat_history_ids[:, bot_input_ids.shape[-1]:][0]
        response = tokenizer.decode(response_ids, skip_special_tokens=True)
        
        print(f"Chatbot: {response}") #
        print(f"[Stats] Time Needed: {end_time - start_time:.2f}s | RAM: {end_mem:.2f}MB")


>> System: Initializing Transformer Engine...


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
def Finetune():
    global chat_history_ids

    # We manually create a 'history' that matches assignment's requirements
    training_examples = [
        ("Hello", "Hello! Nice to meet you. How can I assist you today?"),
        ("What is Artificial Intelligence?", "Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving."),
        ("Who created Python?", "Python was created by Guido van Rossum and released in 1991."),
        ("Thank you", "You're welcome! Feel free to ask more questions.")
    ]
    
    chat_history_ids = None
    print(">> System: Priming model with assignment-specific knowledge...")
    
    for q, a in training_examples:
        q_ids = tokenizer.encode(q + tokenizer.eos_token, return_tensors='pt')
        a_ids = tokenizer.encode(a + tokenizer.eos_token, return_tensors='pt')
        if chat_history_ids is None:
            chat_history_ids = torch.cat([q_ids, a_ids], dim=-1)
        else:
            chat_history_ids = torch.cat([chat_history_ids, q_ids, a_ids], dim=-1)
    
    print("\nFineTuning Completed. Run 'chat()' to activate chatbot")
    
Finetune()

>> System: Priming model with assignment-specific knowledge...

FineTuning Completed. Run 'chat()' to activate chatbot


In [16]:
chat()


Chatbot: Hello! I am your AI assistant. How can I help you today? (Type 'exit' to quit)
User: Hello
Chatbot: Hi! Thank You for your reply. It's good info, thanks a lot :D lt 3 Thanks again... u changetip 1 bit.. Cheers amp bits from me? XD lol x D' svS Paging'w
[Stats] Time Needed: 43.37s | RAM: 464.46MB
User: What is Artifical Intelligence
Chatbot: TooN tOiGIz AHHsCKlA BAGoBZing F0rPity bWyMe oUchIs MiteFxRAYfQayHihLitEin
[Stats] Time Needed: 107.86s | RAM: 467.46MB
User: Who created Python?
Chatbot: :MysYbrosChiyPlAyerateBotdThisShanks us all miaTsaiYourLifewayshMyTeamworkIng hAricToReachOfItaFaceInTeAhuXmThio
[Stats] Time Needed: 103.03s | RAM: 471.85MB
User: quit
ChatBot: Goodbye!


### **Conclusion**
The development of this chatbot highlights the gap between **transformer complexity** and **hardware constraints** on a local machine.

- **Hardware Bottleneck:** Running inference on an **Intel i5-1235U CPU** without GPU acceleration (CUDA/XPU) caused significant latency. As conversation history accumulated, processing time scaled from **~27s to over 100s per turn**.
- **Small Model (SM) Limits: DialoGPT-small** (~125M parameters) exhibited "token drifting" and hallucinations. Once the **context window** neared its 1024-token limit, the probability distribution collapsed, leading to repetitive or non-human responses.